In [ ]:
from stage1_labram_kd_tcn import *
from scipy.stats import pearsonr
import torch
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
@torch.no_grad()
def collect_predictions(model, dataloader, device="cuda"):
    model.eval()
    model.to(device)
    all_pred = []
    all_target = []
    for batch_idx, batch in enumerate(dataloader):
        fmri = batch["fmri_data"].permute(0, 2, 1).float().to(device)
        eeg = batch["eeg_data"].permute(0, 2, 1).float().to(device)
        target = torch.stack([
            batch["alpha_tot_ratio_smoothed"],
        ], dim=1).float().to(device)
        eeg_feats, fmri_feats, x_eeg, x_fmri, logits_eeg, logits_fmri = model(eeg, fmri)
        all_pred.append(logits_fmri.detach().cpu())
        all_target.append(target.detach().cpu())
        
    pred = torch.cat(all_pred, dim=0).numpy()     
    target = torch.cat(all_target, dim=0).numpy()  
    return pred, target


def reconstruct_scan_from_windows(windows, step_size):
    windows = np.asarray(windows)
    n_windows, n_bands, window_size = windows.shape
    full_len = (n_windows - 1) * step_size + window_size 
    out = np.zeros((n_bands, full_len), dtype=np.float32)
    count = np.zeros((1, full_len), dtype=np.float32)
    for i in range(n_windows):
        start = i * step_size
        end = start + window_size
        out[:, start:end] += windows[i]
        count[:, start:end] += 1
    out = out / np.maximum(count, 1.0) 
    return out

def compute_scan_corr_with_pearsonr(pred_scan, target_scan):
    band_names = ["alpha"]
    band_corrs = {}
    for band_idx, band_name in enumerate(band_names):
        corr, p_value = pearsonr(
            pred_scan[band_idx, :],
            target_scan[band_idx, :]
        )
        band_corrs[band_name] = {
            "corr": corr,
            "p_value": p_value,
        }
    mean_corr = np.mean([
        band_corrs["alpha"]["corr"],
    ])
    return band_corrs, mean_corr


def plot_scan_with_pearsonr(scan_name, pred_scan, target_scan, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    band_names = ["alpha"]
    band_corrs, mean_corr = compute_scan_corr_with_pearsonr(pred_scan, target_scan)
    fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
    ax.set_ylim(-0.05, 1.05)
    for band_idx, band_name in enumerate(band_names):
        corr = band_corrs[band_name]["corr"]
        p_value = band_corrs[band_name]["p_value"]
        ax.plot(target_scan[band_idx, :], label="GT", linewidth=2)
        ax.plot(pred_scan[band_idx, :], label="Pred", linewidth=2)
        ax.set_title(
            f"{band_name} corr={corr:.3f}, p={p_value:.2e}"
        )
        ax.legend()
    fig.suptitle(f"{scan_name} | mean corr={mean_corr:.3f}", fontsize=14)
    ax.set_xlabel("fMRI time index")
    plt.tight_layout()
    fig_path = os.path.join(save_dir, f"{scan_name}_pred_vs_gt.png")
    plt.savefig(fig_path, dpi=200)
    plt.close(fig)
    return fig_path, band_corrs, mean_corr


def evaluate_and_visualize_by_scan_with_pearsonr(
    model,
    dataloader,
    dataset,
    save_dir="scan_prediction_outputs",
    device="cuda",
):
    os.makedirs(save_dir, exist_ok=True)
    pred_all, target_all = collect_predictions(
        model=model,
        dataloader=dataloader,
        device=device,
    )
    scan_names = dataset.scan_names
    n_scans = len(scan_names)
    n_total_windows = pred_all.shape[0]
    assert n_total_windows % n_scans == 0, (
        f"Cannot evenly split windows by scan. "
        f"n_total_windows={n_total_windows}, n_scans={n_scans}"
    )

    n_windows_per_scan = n_total_windows // n_scans
    summary = []
    
    for scan_idx, scan_name in enumerate(scan_names):
        
        start_idx = scan_idx * n_windows_per_scan
        
        end_idx = (scan_idx + 1) * n_windows_per_scan
        pred_windows = pred_all[start_idx:end_idx]       
        target_windows = target_all[start_idx:end_idx]   
        pred_scan = reconstruct_scan_from_windows(
            pred_windows,
            step_size=dataset.step_size,
        )
        target_scan = reconstruct_scan_from_windows(
            target_windows,
            step_size=dataset.step_size,
        )

        fig_path, band_corrs, mean_corr = plot_scan_with_pearsonr(
            scan_name=scan_name,
            pred_scan=pred_scan,
            target_scan=target_scan,
            save_dir=save_dir,
        )
        npz_path = os.path.join(save_dir, f"{scan_name}_prediction.npz")
        np.savez(
            npz_path,
            scan_name=scan_name,
            pred_scan=pred_scan,
            target_scan=target_scan,
            pred_windows=pred_windows,
            target_windows=target_windows,
            alpha_corr=band_corrs["alpha"]["corr"],
            mean_corr=mean_corr,
            step_size=dataset.step_size,
            window_size=dataset.window_size,
        )
        row = {
            "scan_name": scan_name,
            "alpha_corr": band_corrs["alpha"]["corr"],
            "mean_corr": mean_corr,
            "fig_path": fig_path,
            "npz_path": npz_path,
        }
        summary.append(row)
        print(
            f"{scan_name}: "
            f"alpha={row['alpha_corr']:.3f}, "
            f"mean={row['mean_corr']:.3f}"
        )
    return summary

In [ ]:
vu_healthy_test_dataset = VigilanceDataset1024(dataset_name="vu_healthy", window_size=50, step_size=50, vigilance_threshold=-25, split_set="test")
vu_healthy_test_dataset.scan_names

In [ ]:
model_test = LitEEGPTCausal(pretrained_ckpt_path= ) #TODO

In [ ]:
vu_healthy_test_loader = DataLoader(
    vu_healthy_test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
)

summary = evaluate_and_visualize_by_scan_with_pearsonr(
    model=model_test,
    dataloader=vu_healthy_test_loader,
    dataset=vu_healthy_test_dataset,
    save_dir=, #TODO
    device="cuda",
)

In [ ]:
alpha_mean = np.mean([row["alpha_corr"] for row in summary])
print("--------------------------------------------------")
print(f"Mean alpha corr: {alpha_mean:.4f}")
print("--------------------------------------------------")

In [ ]:
vu_pat_zeroshot_dataset = VigilanceDataset1024(dataset_name="vu_pat", window_size=50, step_size=50, vigilance_threshold=-25, split_set="zero_shot")
vu_pat_zeroshot_dataset.scan_names

In [ ]:
vu_pat_zeroshot_loader = DataLoader(
    vu_pat_zeroshot_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
)

summary = evaluate_and_visualize_by_scan_with_pearsonr(
    model=model_test,
    dataloader=vu_pat_zeroshot_loader,
    dataset=vu_pat_zeroshot_dataset,
    save_dir= , #TODO
    device="cuda",
)

In [ ]:
alpha_mean = np.mean([row["alpha_corr"] for row in summary])
print("--------------------------------------------------")
print(f"Mean alpha corr: {alpha_mean:.4f}")
print("--------------------------------------------------")